# 05 - Inference Profiling (Colab)

Measures VRAM, latency, and throughput for all 7 checkpoints: Base, SFT-R4/8/16 (merged FP16), Quantized-R4/8/16 (GPTQ 4-bit). Produces `eval/results/{checkpoint}/timing.json`.

**Before running:** `Runtime > Change runtime type > T4 GPU`. **Requires Phase 5's outputs** at `outputs/merged_r{rank}/` and `outputs/quantized_r{rank}/` — mount the same Drive folder used in `03_qlora_training.ipynb`/`04_merge_quantize.ipynb` to find them.

In [ ]:
REPO_URL = ""  # e.g. "https://<TOKEN>@github.com/Shhaurya17/Efficient-Small-Language-Model-Adaptation-Quantization-Benchmark.git"
USE_DRIVE = True
DRIVE_WORKDIR = "/content/drive/MyDrive/efficient-slm-benchmark"

import os

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_WORKDIR, exist_ok=True)

REPO_DIR = os.path.join(DRIVE_WORKDIR, "repo") if USE_DRIVE else "/content/efficient-slm-benchmark"

if REPO_URL and not os.path.exists(os.path.join(REPO_DIR, ".git")):
    !git clone -q {REPO_URL} {REPO_DIR}

HAVE_REPO = os.path.exists(os.path.join(REPO_DIR, "configs", "model.yaml"))
OUTPUT_ROOT = os.path.join(REPO_DIR, "outputs") if HAVE_REPO else os.path.join(DRIVE_WORKDIR, "outputs")
RESULTS_ROOT = os.path.join(REPO_DIR, "eval", "results") if HAVE_REPO else os.path.join(DRIVE_WORKDIR, "eval_results")
os.makedirs(RESULTS_ROOT, exist_ok=True)
print("Repo available:", HAVE_REPO)
print("Output root:", OUTPUT_ROOT)

In [ ]:
%%capture
!pip install -q transformers>=4.44.0 accelerate>=0.33.0 optimum>=1.21.0 auto-gptq>=0.7.1 pyyaml matplotlib

In [ ]:
import sys
import yaml

if HAVE_REPO:
    sys.path.insert(0, os.path.join(REPO_DIR, "src"))
    with open(os.path.join(REPO_DIR, "configs", "model.yaml")) as f:
        model_config = yaml.safe_load(f)
else:
    model_config = {"model_name": "Qwen/Qwen2.5-1.5B-Instruct", "torch_dtype": "float16"}

from efficient_slm.inference.profiler import profile_checkpoint

CHECKPOINTS = {"base": model_config["model_name"]}
for rank in (4, 8, 16):
    CHECKPOINTS[f"sft_r{rank}"] = os.path.join(OUTPUT_ROOT, f"merged_r{rank}")
    CHECKPOINTS[f"quantized_r{rank}"] = os.path.join(OUTPUT_ROOT, f"quantized_r{rank}")

for name, path_ in CHECKPOINTS.items():
    exists = name == "base" or os.path.exists(path_)
    print(f"{name}: {path_} (available: {exists})")

## Profile each checkpoint

In [ ]:
import gc
import json

import torch

all_timing = {}
for name, ckpt_path in CHECKPOINTS.items():
    if name != "base" and not os.path.exists(ckpt_path):
        print(f"Skipping {name}: not found at {ckpt_path}")
        continue

    print(f"=== Profiling {name} ===")
    report, model, tokenizer = profile_checkpoint(ckpt_path, torch_dtype=model_config.get("torch_dtype", "float16"))
    all_timing[name] = report

    checkpoint_results_dir = os.path.join(RESULTS_ROOT, name)
    os.makedirs(checkpoint_results_dir, exist_ok=True)
    with open(os.path.join(checkpoint_results_dir, "timing.json"), "w") as f:
        json.dump(report, f, indent=2)

    print(report)
    del model
    gc.collect()
    torch.cuda.empty_cache()

all_timing

## VRAM vs latency

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 6))
for name, report in all_timing.items():
    vram = report["vram"]["peak_vram_gb"]
    latency = report["latency"]["ms_per_token"]
    if vram is None:
        continue
    ax.scatter(vram, latency, s=80)
    ax.annotate(name, (vram, latency), textcoords="offset points", xytext=(5, 5))
ax.set_xlabel("Peak VRAM (GB)")
ax.set_ylabel("Latency (ms/token, batch=1)")
ax.set_title("VRAM vs Latency Across Checkpoints")
plt.tight_layout()
figures_dir = os.path.join(REPO_DIR, "figures") if HAVE_REPO else os.path.join(DRIVE_WORKDIR, "figures")
os.makedirs(figures_dir, exist_ok=True)
plt.savefig(os.path.join(figures_dir, "profiling_vram_latency.png"), dpi=150)
plt.show()